# Claude Agents SDK - Proper Setup and Usage

## Overview
This notebook demonstrates proper usage of the Claude Agents SDK in Cursor IDE with:
- ??? Proper environment setup with uv
- ??? Async/await patterns
- ??? Configuration management via .env
- ??? Logging with loguru
- ? Streaming responses
- ? Error handling patterns

In [ ]:
# Environment and imports setup
import os
import sys
import asyncio
from pathlib import Path
from dataclasses import dataclass
from typing import AsyncGenerator, Optional

# Third-party imports
import anyio
from loguru import logger as log
from dotenv import load_dotenv
from claude_agent_sdk import query, ClaudeAgentOptions

# Load environment variables
load_dotenv()

In [ ]:
# Configuration class following user conventions
@dataclass
class Config:
    """Configuration for Claude Agents SDK notebook."""
    # Logging
    LOG_LEVEL: str = "INFO"  # Toggle between DEBUG/INFO
    LOG_FORMAT: str = "{time:YYYY-MM-DD HH:mm:ss} | {level} | {message}"
    
    # API Configuration
    ANTHROPIC_API_KEY: Optional[str] = os.getenv("ANTHROPIC_API_KEY")
    
    # Agent Options
    SYSTEM_PROMPT: str = "You are a helpful assistant focused on coding tasks."
    MAX_TOKENS: int = 1000
    
    def log_init(self):
        """Initialize loguru with color formatting."""
        log.remove()  # Remove default handler
        log.add(sys.stderr, format=self.LOG_FORMAT, level=self.LOG_LEVEL)

# Initialize config
CONFIG = Config()
CONFIG.log_init()

log.info("?? Claude Agents SDK notebook initialized")
log.debug(f"Config: {CONFIG}")

In [ ]:
# Basic query example
async def basic_query_example():
    """Demonstrate basic Claude Agent SDK query."""
    log.info("Running basic query example...")
    
    try:
        async for message in query(prompt="What is 2 + 2? Explain briefly."):
            print(f"Response: {message}")
    except Exception as e:
        log.error(f"Basic query failed: {e}")
        return False
    
    log.success("? Basic query completed")
    return True

# Run in notebook
result = await basic_query_example()
print(f"Basic query result: {result}")

In [ ]:
# Advanced query with options
async def advanced_query_example():
    """Demonstrate Claude Agent SDK with custom options."""
    log.info("Running advanced query with custom options...")
    
    options = ClaudeAgentOptions(
        system_prompt=CONFIG.SYSTEM_PROMPT,
        max_tokens=CONFIG.MAX_TOKENS,
        # Add more options as needed
    )
    
    prompt = """
    Create a simple Python function that calculates the factorial of a number.
    Include type hints and a docstring.
    """
    
    try:
        response_chunks = []
        async for message in query(prompt=prompt, options=options):
            response_chunks.append(message)
            print(f"Chunk: {message[:100]}...")  # Show first 100 chars
        
        full_response = "".join(response_chunks)
        log.info(f"Full response length: {len(full_response)} characters")
        
        return full_response
    except Exception as e:
        log.error(f"Advanced query failed: {e}")
        return None

# Run advanced example
advanced_result = await advanced_query_example()
if advanced_result:
    print("\n" + "="*50)
    print("FULL RESPONSE:")
    print(advanced_result)
    print("="*50)

In [ ]:
# Helper functions for common operations
class AgentHelpers:
    """Helper class for common agent operations."""
    
    @staticmethod
    async def code_generation_query(task: str) -> Optional[str]:
        """Generate code for a specific task."""
        prompt = f"""
        Generate Python code for the following task:
        {task}
        
        Requirements:
        - Use type hints
        - Include docstrings
        - Follow Python best practices
        - Add error handling where appropriate
        """
        
        try:
            result = []
            async for chunk in query(prompt=prompt):
                result.append(chunk)
            return "".join(result)
        except Exception as e:
            log.error(f"Code generation failed: {e}")
            return None
    
    @staticmethod
    async def explain_code_query(code: str) -> Optional[str]:
        """Explain what a piece of code does."""
        prompt = f"""
        Explain what this Python code does:
        
        ```python
        {code}
        ```
        
        Provide a clear, concise explanation.
        """
        
        try:
            result = []
            async for chunk in query(prompt=prompt):
                result.append(chunk)
            return "".join(result)
        except Exception as e:
            log.error(f"Code explanation failed: {e}")
            return None

# Test helper functions
helpers = AgentHelpers()

# Generate code example
code_task = "A function that reads a CSV file and returns a pandas DataFrame"
generated_code = await helpers.code_generation_query(code_task)

if generated_code:
    print("GENERATED CODE:")
    print(generated_code)
    print("\n" + "-"*50 + "\n")
    
    # Explain the generated code
    explanation = await helpers.explain_code_query(generated_code)
    if explanation:
        print("CODE EXPLANATION:")
        print(explanation)

In [ ]:
# Environment validation and diagnostics
def validate_environment():
    """Validate the current environment setup."""
    log.info("?? Validating environment...")
    
    checks = {
        "Python version": sys.version,
        "Current working directory": os.getcwd(),
        "Virtual environment": os.environ.get('VIRTUAL_ENV', 'Not detected'),
        "ANTHROPIC_API_KEY set": bool(CONFIG.ANTHROPIC_API_KEY),
        "Claude SDK available": True,  # If we got here, it's imported
        "Loguru working": True,  # If we got here, logging works
    }
    
    print("Environment Validation Results:")
    print("=" * 40)
    
    for check, result in checks.items():
        status = "?" if result else "?"
        print(f"{status} {check}: {result}")
    
    # Check if we're in Jupyter
    try:
        from IPython import get_ipython
        if get_ipython() is not None:
            print("? Running in Jupyter environment")
            print(f"? IPython version: {get_ipython().__class__.__name__}")
        else:
            print("? Not running in Jupyter")
    except ImportError:
        print("? IPython not available")
    
    print("=" * 40)
    log.success("Environment validation complete")

# Run validation
validate_environment()

## Next Steps

### ? Completed Setup:
- uv package manager installed
- Virtual environment created
- JupyterLab and dependencies installed
- Claude Agents SDK properly configured
- Logging with loguru
- Environment validation

### ? TODO:
- Configure ANTHROPIC_API_KEY in `.env` file
- Test streaming responses
- Add more advanced examples
- Implement error recovery patterns

### ?? Cursor IDE Integration:
- This notebook should now work properly in Cursor IDE
- Use `Cmd/Ctrl + Shift + P` ? "Python: Select Interpreter" ? Choose the `.venv/bin/python`
- JupyterLab should detect the kernel automatically

In [ ]:
# Final test - demonstrate everything works
async def integration_test():
    """Run integration test to verify everything works."""
    log.info("?? Running integration test...")
    
    tests_passed = 0
    total_tests = 3
    
    # Test 1: Basic query
    try:
        result = []
        async for chunk in query("Say hello and confirm you're working."):
            result.append(chunk)
        if result:
            log.success("? Test 1: Basic query - PASSED")
            tests_passed += 1
        else:
            log.error("? Test 1: Basic query - FAILED (empty response)")
    except Exception as e:
        log.error(f"? Test 1: Basic query - FAILED ({e})")
    
    # Test 2: Config access
    try:
        if CONFIG.LOG_LEVEL and CONFIG.SYSTEM_PROMPT:
            log.success("? Test 2: Config access - PASSED")
            tests_passed += 1
        else:
            log.error("? Test 2: Config access - FAILED")
    except Exception as e:
        log.error(f"? Test 2: Config access - FAILED ({e})")
    
    # Test 3: Helper functions
    try:
        helpers = AgentHelpers()
        if hasattr(helpers, 'code_generation_query'):
            log.success("? Test 3: Helper functions - PASSED")
            tests_passed += 1
        else:
            log.error("? Test 3: Helper functions - FAILED")
    except Exception as e:
        log.error(f"? Test 3: Helper functions - FAILED ({e})")
    
    # Summary
    success_rate = (tests_passed / total_tests) * 100
    print(f"\n?? Integration Test Results: {tests_passed}/{total_tests} tests passed ({success_rate:.0f}%)")
    
    if tests_passed == total_tests:
        log.success("?? All tests passed! Environment is ready.")
        return True
    else:
        log.warning(f"?? {total_tests - tests_passed} test(s) failed. Check configuration.")
        return False

# Run final integration test
test_result = await integration_test()
print(f"\n?? Setup complete! Success: {test_result}")